# Data Cleaning

This step comes first in the data analysis workflow. Once the dataset is cleaned and reliable, it can then be explored through exploratory data analysis (EDA) to uncover patterns and insights for feature engineering and decision-making.

The dataset used in this notebook is sourced from [Kaggle](https://www.kaggle.com/datasets/imtkaggleteam/diabetes) and contains health-related attributes for diabetes analysis.


**1. Import Libraries**

In [0]:
# Import libraries
from pyspark.sql.functions import col, mean,  median, expr, sum

**2. Load Data**

In [0]:
# Load data to dataframe and display 
df = spark.table('workspace.default.diabetes')
display(df.limit(5))

id,chol,stab.glu,hdl,ratio,glyhb,location,age,gender,height,weight,frame,bp.1s,bp.1d,bp.2s,bp.2d,waist,hip,time.ppn
1000,203,82,56,3.5999999046,4.3099999428,Buckingham,46,female,62,121,medium,118,59,null,null,29,38,720
1001,165,97,24,6.9000000954,4.4400000572,Buckingham,29,female,64,218,large,112,68,null,null,46,48,360
1002,228,92,37,6.1999998093,4.6399998665,Buckingham,58,female,61,256,large,190,92,185,92,49,57,180
1003,78,93,12,6.5,4.6300001144,Buckingham,67,male,67,119,large,110,50,null,null,33,38,480
1005,249,90,28,8.8999996185,7.7199997902,Buckingham,64,male,68,183,medium,138,80,null,null,44,41,300


**3. Fix Data Types and Rename Fields**

Correct data types ensure accurate calculations and analysis, while clear, descriptive column names make the dataset easier to read, understand, and share.


In [0]:
data = df.select(
    col("id").astype("int").alias("patient_id"),
    col("chol").astype("float").alias("cholesterol_total"),
    col("`stab.glu`").astype("float").alias("stabilized_glucose"),
    col("hdl").astype("float").alias("hdl_cholesterol"),
    col("ratio").astype("float").alias("chol_hdl_ratio"),
    col("glyhb").astype("float").alias("hba1c_percent"),
    col("location").astype("string").alias("location"),
    col("age").astype("int"),
    col("gender").astype("string"),
    col("height").astype("float").alias("height_inches"),
    col("weight").astype("float").alias("weight_pounds"),
    col("frame").astype("string").alias("body_frame"),
    col("`bp.1s`").astype("float").alias("systolic_bp_1"),
    col("`bp.1d`").astype("float").alias("diastolic_bp_1"),
    col("`bp.2s`").astype("float").alias("systolic_bp_2"),
    col("`bp.2d`").astype("float").alias("diastolic_bp_2"),
    col("waist").astype("float").alias("waist_inches"),
    col("hip").astype("float").alias("hip_inches"),
    col("`time.ppn`").astype("int").alias("minutes_post_meal")
)

**4. Handling Missing Values**

- There is no single rule for filling or predicting missing values.
- The number of missing values may determine the method of handling nulls.  
- The selection of a method depends on the dataset and the specific problem at hand. 
- One may select an approach considered appropriate and train the model.  
- If the model’s evaluation metrics are unsatisfactory, one may use a different strategy for handling missing values and re-evaluate the model again.
- Thresholds may be defined as well such as drop columns with >50% missing values.

In [0]:
# Loop through all columns in the DataFrame and count nulls
for c in data.columns:
    print(c, data.filter(col(c).isNull()).count())

patient_id 0
cholesterol_total 1
stabilized_glucose 0
hdl_cholesterol 1
chol_hdl_ratio 1
hba1c_percent 13
location 0
age 0
gender 0
height_inches 5
weight_pounds 1
body_frame 12
systolic_bp_1 5
diastolic_bp_1 5
systolic_bp_2 262
diastolic_bp_2 262
waist_inches 2
hip_inches 2
minutes_post_meal 3


The fields `systolic_bp_2` and `diastolic_bp_2` contain more than 50% missing values.  
According to Microsoft’s guidance on [Clean Missing Data](https://learn.microsoft.com/en-us/azure/machine-learning/component-reference/clean-missing-data), imputing such extensive gaps can introduce bias and reduce model reliability.  
Dropping these fields is a safer choice to maintain data integrity and avoid misleading insights.


In [0]:
# drop systolic_bp_2 and diastolic_bp_2
data_new = data.drop("systolic_bp_2", "diastolic_bp_2")

The field `hba1c_percent` contains 13 missing values. I will apply an imputation method to address them.

In [0]:
display(data_new.select('hba1c_percent').describe())

summary,hba1c_percent
count,390
mean,5.589769236246744
stddev,2.2425948419917026
min,2.68
max,16.11


In [0]:
# Calculate mean and median
mean_val = data_new.select(mean(col("hba1c_percent"))).first()[0]
median_val = data_new.approxQuantile("hba1c_percent", [0.5], 0.01)[0]

print("Mean:", mean_val)
print("Median:", median_val)

Mean: 5.589769236246744
Median: 4.840000152587891


The distribution of `hba1c_percent` appears to be skewed, making the median a better option for imputation.

In [0]:
# Impute missing values with the median
data_cleaned = data_new.fillna({"hba1c_percent": median_val})

In [0]:
# display body frame missing values
display(data_cleaned.select('body_frame').groupBy("body_frame").count())

body_frame,count
medium,184
large,103
small,104
null,12


Databricks visualization. Run in Databricks to view.

The field `body_frame` contains missing values. Since this variable is categorical, the mode was selected as the imputation method. The mode represents the most frequently occurring category, making it the most reliable choice for filling missing entries in categorical data while preserving the dominant pattern in the dataset.


In [0]:
# Compute mode for body_frame
mode_val = (data_cleaned.groupBy("body_frame").count().orderBy("count", ascending=False).first()["body_frame"])

print("Mode value for body_frame:", mode_val)

# Impute missing values with the mode
data_clean = data_cleaned.fillna({"body_frame": mode_val})

Mode value for body_frame: medium


In [0]:
# Loop through all columns in the DataFrame and count nulls
for c in data_clean.columns:
    print(c, data_clean.filter(col(c).isNull()).count())

patient_id 0
cholesterol_total 1
stabilized_glucose 0
hdl_cholesterol 1
chol_hdl_ratio 1
hba1c_percent 0
location 0
age 0
gender 0
height_inches 5
weight_pounds 1
body_frame 0
systolic_bp_1 5
diastolic_bp_1 5
waist_inches 2
hip_inches 2
minutes_post_meal 3


In [0]:
# Drop all rows with any missing values
data_final = data_clean.dropna()

#Check if any missing vales left
data_final.select([sum(col(c).isNull().cast("int")).alias(c) for c in data_final.columns]).display()

patient_id,cholesterol_total,stabilized_glucose,hdl_cholesterol,chol_hdl_ratio,hba1c_percent,location,age,gender,height_inches,weight_pounds,body_frame,systolic_bp_1,diastolic_bp_1,waist_inches,hip_inches,minutes_post_meal
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
#display final count
display(data_final.count())

388

The dataset `data_final` will serve as the input for exploratory data analysis, providing a complete and consistent foundation for uncovering patterns, relationships, and insights.


In [0]:
# Write the final dataset to the storage
data_final.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("diabetes_data_cleaned")